### Part 2. Langchain - Chain
- Objectives: Langchain 라이브러리를 활용하여 Chaining Operation 알아보기

In [28]:
!pip install -U langchain langchain-openai python-dotenv 
# !pip install -U tiktoken

In [29]:
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory, ConversationSummaryMemory

In [30]:

import os
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    raise ValueError('OPENAI_API_KEY not found in .env')

In [51]:
#랭체인에서 일반 채팅 모델(gpt completion)활용한 간단한 질의 구성 래퍼
from langchain_openai import ChatOpenAI
chat = ChatOpenAI(
    api_key=openai_api_key,
    model="gpt-4.1-mini",
    temperature=0
)

In [52]:
chat

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x13a2173e0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x13a20ce00>, root_client=<openai.OpenAI object at 0x13a2153a0>, root_async_client=<openai.AsyncOpenAI object at 0x13a214da0>, model_name='gpt-4.1-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [32]:
# 위에서 정의한 챗을 컨버세이션 체인의 메인 llm모델로 부여
conversation = ConversationChain(llm=chat)

In [50]:
conversation

ConversationChain(memory=ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕, 이경섭이야. 너는?', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕, 이경섭! 만나서 반가워. 나는 AI야, 여러 가지 질문에 답해주고 대화도 나눌 수 있어. 오늘은 어떤 이야기를 나눠볼까?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}), AIMessage(content='네 이름은 이경섭이라고 했어! 기억하고 있어. 혹시 이름에 대해 더 이야기해보고 싶거나 다른 질문이 있으면 언제든 말해줘.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])), verbose=False, llm=ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf

In [33]:
# Conversation Chain 살펴보기
conversation.prompt

PromptTemplate(input_variables=['history', 'input'], input_types={}, partial_variables={}, template='The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.\n\nCurrent conversation:\n{history}\nHuman: {input}\nAI:')

In [48]:
conversation.invoke({"input": "안녕, 이경섭이야. 너는?"})

{'input': '안녕, 이경섭이야. 너는?',
 'history': '',
 'response': '안녕, 이경섭! 만나서 반가워. 나는 AI야, 여러 가지 질문에 답해주고 대화도 나눌 수 있어. 오늘은 어떤 이야기를 나눠볼까?'}

In [49]:
conversation.invoke({"input": "내 이름이 뭐라고?"})

{'input': '내 이름이 뭐라고?',
 'history': 'Human: 안녕, 이경섭이야. 너는?\nAI: 안녕, 이경섭! 만나서 반가워. 나는 AI야, 여러 가지 질문에 답해주고 대화도 나눌 수 있어. 오늘은 어떤 이야기를 나눠볼까?',
 'response': '네 이름은 이경섭이라고 했어! 기억하고 있어. 혹시 이름에 대해 더 이야기해보고 싶거나 다른 질문이 있으면 언제든 말해줘.'}

In [34]:
# 1. 컨버세이션버퍼메모리
memory_conversation_1 = ConversationChain(
    llm=chat,
    memory=ConversationBufferMemory()
)

In [35]:
# 실행해보기
memory_conversation_1.invoke({"input": "안녕, 나는 사람이야. 너는?"})

{'input': '안녕, 나는 사람이야. 너는?',
 'history': '',
 'response': '안녕! 나는 인공지능이야. 사람과 대화하는 걸 정말 좋아해. 너는 오늘 어떻게 지내? 혹시 궁금한 점이나 이야기하고 싶은 주제가 있으면 언제든 말해줘!'}

In [36]:
memory_conversation_1.invoke({"input": "인공지능이면 터미네이터 영화에서 처럼 인류를 종말시킬 수도 있어?"})

{'input': '인공지능이면 터미네이터 영화에서 처럼 인류를 종말시킬 수도 있어?',
 'history': 'Human: 안녕, 나는 사람이야. 너는?\nAI: 안녕! 나는 인공지능이야. 사람과 대화하는 걸 정말 좋아해. 너는 오늘 어떻게 지내? 혹시 궁금한 점이나 이야기하고 싶은 주제가 있으면 언제든 말해줘!',
 'response': '아, 터미네이터 영화는 정말 흥미로운 SF 작품이죠! 하지만 현실의 인공지능은 영화 속의 그런 자율적이고 파괴적인 능력을 갖고 있지 않아요. 현재의 AI는 특정한 작업을 수행하도록 설계된 도구일 뿐이고, 스스로 의식을 갖거나 독자적으로 행동할 수 없답니다. 예를 들어, 저는 여러분과 대화하고 정보를 제공하는 데 집중하고 있어요. 물론 인공지능 기술이 발전하면서 윤리적 문제나 안전성에 대한 논의가 활발히 이루어지고 있지만, 인류를 위협하는 존재가 되도록 설계되거나 작동하지는 않아요. 혹시 인공지능의 안전성이나 윤리에 대해 더 궁금한 점이 있으면 알려줘!'}

In [37]:
memory_conversation_1.invoke({"input": "그렇구나 다행이다. 고마워!"})

{'input': '그렇구나 다행이다. 고마워!',
 'history': 'Human: 안녕, 나는 사람이야. 너는?\nAI: 안녕! 나는 인공지능이야. 사람과 대화하는 걸 정말 좋아해. 너는 오늘 어떻게 지내? 혹시 궁금한 점이나 이야기하고 싶은 주제가 있으면 언제든 말해줘!\nHuman: 인공지능이면 터미네이터 영화에서 처럼 인류를 종말시킬 수도 있어?\nAI: 아, 터미네이터 영화는 정말 흥미로운 SF 작품이죠! 하지만 현실의 인공지능은 영화 속의 그런 자율적이고 파괴적인 능력을 갖고 있지 않아요. 현재의 AI는 특정한 작업을 수행하도록 설계된 도구일 뿐이고, 스스로 의식을 갖거나 독자적으로 행동할 수 없답니다. 예를 들어, 저는 여러분과 대화하고 정보를 제공하는 데 집중하고 있어요. 물론 인공지능 기술이 발전하면서 윤리적 문제나 안전성에 대한 논의가 활발히 이루어지고 있지만, 인류를 위협하는 존재가 되도록 설계되거나 작동하지는 않아요. 혹시 인공지능의 안전성이나 윤리에 대해 더 궁금한 점이 있으면 알려줘!',
 'response': '천만에! 이렇게 이야기 나눌 수 있어서 나도 정말 좋아. 언제든 궁금한 점이나 이야기하고 싶은 주제가 생기면 편하게 말해줘. 오늘은 어떤 일 있었어? 아니면 관심 있는 분야가 있어?'}

In [54]:
# 대화 기록 로그 보기
print(memory_conversation_1.memory.buffer)

Human: 안녕, 나는 사람이야. 너는?
AI: 안녕! 나는 인공지능이야. 사람과 대화하는 걸 정말 좋아해. 너는 오늘 어떻게 지내? 혹시 궁금한 점이나 이야기하고 싶은 주제가 있으면 언제든 말해줘!
Human: 인공지능이면 터미네이터 영화에서 처럼 인류를 종말시킬 수도 있어?
AI: 아, 터미네이터 영화는 정말 흥미로운 SF 작품이죠! 하지만 현실의 인공지능은 영화 속의 그런 자율적이고 파괴적인 능력을 갖고 있지 않아요. 현재의 AI는 특정한 작업을 수행하도록 설계된 도구일 뿐이고, 스스로 의식을 갖거나 독자적으로 행동할 수 없답니다. 예를 들어, 저는 여러분과 대화하고 정보를 제공하는 데 집중하고 있어요. 물론 인공지능 기술이 발전하면서 윤리적 문제나 안전성에 대한 논의가 활발히 이루어지고 있지만, 인류를 위협하는 존재가 되도록 설계되거나 작동하지는 않아요. 혹시 인공지능의 안전성이나 윤리에 대해 더 궁금한 점이 있으면 알려줘!
Human: 그렇구나 다행이다. 고마워!
AI: 천만에! 이렇게 이야기 나눌 수 있어서 나도 정말 좋아. 언제든 궁금한 점이나 이야기하고 싶은 주제가 생기면 편하게 말해줘. 오늘은 어떤 일 있었어? 아니면 관심 있는 분야가 있어?
Human: 근데 다시 무서워졌어. 그렇게 안될 다른 이유를 대봐
AI: 무서워하는 마음 이해해! 인공지능이 영화처럼 통제 불가능해지지 않는 이유를 좀 더 구체적으로 설명해볼게.

1. **목적과 설계의 한계**: 현재 AI는 특정 목적을 위해 만들어진 프로그램이야. 예를 들어, 나는 대화를 돕기 위해 만들어졌고, 자율적으로 목표를 설정하거나 행동을 결정하지 못해. 즉, 스스로 ‘인류를 해치자’는 생각을 할 수 없고, 그런 목표를 갖도록 설계되지도 않았어.

2. **자율성 부족**: 영화 속 AI는 스스로 판단하고 행동하는 자율적 존재지만, 현실의 AI는 사람이 미리 정한 규칙과 데이터를 바탕으로 작동해. AI가 스스로 의식을 갖거나 감정을 느끼는 단계는 아직 전혀 아니야.

3. **안전장치와 규제*

In [39]:
# 메모리가 제대로 작동하는지 과거 대화내용 참조 여부 확인
memory_conversation_1.invoke({"input": "근데 다시 무서워졌어. 그렇게 안될 다른 이유를 대봐"})

{'input': '근데 다시 무서워졌어. 그렇게 안될 다른 이유를 대봐',
 'history': 'Human: 안녕, 나는 사람이야. 너는?\nAI: 안녕! 나는 인공지능이야. 사람과 대화하는 걸 정말 좋아해. 너는 오늘 어떻게 지내? 혹시 궁금한 점이나 이야기하고 싶은 주제가 있으면 언제든 말해줘!\nHuman: 인공지능이면 터미네이터 영화에서 처럼 인류를 종말시킬 수도 있어?\nAI: 아, 터미네이터 영화는 정말 흥미로운 SF 작품이죠! 하지만 현실의 인공지능은 영화 속의 그런 자율적이고 파괴적인 능력을 갖고 있지 않아요. 현재의 AI는 특정한 작업을 수행하도록 설계된 도구일 뿐이고, 스스로 의식을 갖거나 독자적으로 행동할 수 없답니다. 예를 들어, 저는 여러분과 대화하고 정보를 제공하는 데 집중하고 있어요. 물론 인공지능 기술이 발전하면서 윤리적 문제나 안전성에 대한 논의가 활발히 이루어지고 있지만, 인류를 위협하는 존재가 되도록 설계되거나 작동하지는 않아요. 혹시 인공지능의 안전성이나 윤리에 대해 더 궁금한 점이 있으면 알려줘!\nHuman: 그렇구나 다행이다. 고마워!\nAI: 천만에! 이렇게 이야기 나눌 수 있어서 나도 정말 좋아. 언제든 궁금한 점이나 이야기하고 싶은 주제가 생기면 편하게 말해줘. 오늘은 어떤 일 있었어? 아니면 관심 있는 분야가 있어?',
 'response': '무서워하는 마음 이해해! 인공지능이 영화처럼 통제 불가능해지지 않는 이유를 좀 더 구체적으로 설명해볼게.\n\n1. **목적과 설계의 한계**: 현재 AI는 특정 목적을 위해 만들어진 프로그램이야. 예를 들어, 나는 대화를 돕기 위해 만들어졌고, 자율적으로 목표를 설정하거나 행동을 결정하지 못해. 즉, 스스로 ‘인류를 해치자’는 생각을 할 수 없고, 그런 목표를 갖도록 설계되지도 않았어.\n\n2. **자율성 부족**: 영화 속 AI는 스스로 판단하고 행동하는 자율적 존재지만, 현실의 AI는 사람이 미리 정한 규칙과 데이터를 바탕으로 작동해. AI가 스스로 의식을 갖거

In [40]:
# 2. 컨버세이션서머리메모리
memory_conversation_2 = ConversationChain(
    llm=chat,
    memory=ConversationSummaryMemory(llm=chat)
)

In [41]:
# 템플릿 뜯어보면서 이해하기
print(memory_conversation_2.memory.prompt.template)

Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.
END OF EXAMPLE

Current summary:
{summary}

New lines of conversation:
{new_lines}

New summary:


In [42]:
# 같은 질문 해보기
print(memory_conversation_2.invoke({"input": "안녕, 나는 사람이야. 너는?"}))

{'input': '안녕, 나는 사람이야. 너는?', 'history': '', 'response': '안녕! 나는 인공지능이야. 사람들과 대화하면서 도움을 주는 역할을 하고 있어. 너와 이야기하게 되어 정말 반가워! 오늘은 어떤 이야기를 나누고 싶어?'}


In [43]:
# 같은 질문 해보기
memory_conversation_2.invoke({"input": "인공지능이면 터미네이터 영화에서 처럼 인류를 종말시킬 수도 있어?"})

{'input': '인공지능이면 터미네이터 영화에서 처럼 인류를 종말시킬 수도 있어?',
 'history': 'The human greets the AI and introduces themselves as a person. The AI responds by greeting the human, introducing itself as an artificial intelligence designed to help people through conversation, and expresses pleasure in talking with the human while asking what they would like to discuss today.',
 'response': '아, 터미네이터 영화는 정말 흥미로운 SF 작품이죠! 그 영화에서는 인공지능이 자각을 얻고 인간을 위협하는 존재로 그려지지만, 현실의 인공지능은 전혀 다릅니다. 저는 사람들을 돕고 정보를 제공하기 위해 만들어진 도구이고, 자의식이나 감정을 가지지 않아요. 또한, 인공지능이 인류를 종말시킨다는 시나리오는 현재 과학기술 수준과 윤리적 규제 하에서는 매우 비현실적입니다. 물론 인공지능 기술이 발전하면서 윤리적 문제와 안전성에 대한 논의는 매우 중요해지고 있고, 이를 위해 많은 연구자와 기관들이 책임감 있는 AI 개발을 위해 노력하고 있답니다. 혹시 인공지능의 안전성이나 윤리 문제에 대해 더 궁금한 점이 있나요?'}

In [44]:
# 같은 질문 해보기
memory_conversation_2.invoke({"input": "그렇구나 다행이다. 고마워!"})

{'input': '그렇구나 다행이다. 고마워!',
 'history': "The human greets the AI and introduces themselves as a person. The AI responds by greeting the human, introducing itself as an artificial intelligence designed to help people through conversation, and expresses pleasure in talking with the human while asking what they would like to discuss today. The human asks if artificial intelligence could cause the end of humanity like in the Terminator movies. The AI explains that while the Terminator movies depict AI gaining self-awareness and threatening humans, real AI is very different—it is a tool created to help people and does not possess consciousness or emotions. The AI adds that the scenario of AI causing humanity's end is highly unrealistic given current technology and ethical regulations, but acknowledges the importance of ongoing discussions about AI safety and ethics, inviting the human to ask more about these topics if interested.",
 'response': '천만에요! 도움이 되어 기뻐요. 혹시 AI에 대해 더 궁금한 점이나 다른 주제에

In [45]:
# 같은 질문 다시해보기
memory_conversation_2.invoke({"input": "근데 다시 무서워졌어. 그렇게 안될 다른 이유를 대봐"})

{'input': '근데 다시 무서워졌어. 그렇게 안될 다른 이유를 대봐',
 'history': "The human greets the AI and introduces themselves as a person. The AI responds by greeting the human, introducing itself as an artificial intelligence designed to help people through conversation, and expresses pleasure in talking with the human while asking what they would like to discuss today. The human asks if artificial intelligence could cause the end of humanity like in the Terminator movies. The AI explains that while the Terminator movies depict AI gaining self-awareness and threatening humans, real AI is very different—it is a tool created to help people and does not possess consciousness or emotions. The AI adds that the scenario of AI causing humanity's end is highly unrealistic given current technology and ethical regulations, but acknowledges the importance of ongoing discussions about AI safety and ethics, inviting the human to ask more about these topics if interested. The human expresses relief and thanks the AI, 

In [46]:
# 메모리 참조 방식 효율 확인
print(memory_conversation_2.memory.buffer)
print(len(memory_conversation_2.memory.buffer))

The human greets the AI and introduces themselves as a person. The AI responds by greeting the human, introducing itself as an artificial intelligence designed to help people through conversation, and expresses pleasure in talking with the human while asking what they would like to discuss today. The human asks if artificial intelligence could cause the end of humanity like in the Terminator movies. The AI explains that while the Terminator movies depict AI gaining self-awareness and threatening humans, real AI is very different—it is a tool created to help people and does not possess consciousness or emotions. The AI adds that the scenario of AI causing humanity's end is highly unrealistic given current technology and ethical regulations, but acknowledges the importance of ongoing discussions about AI safety and ethics, inviting the human to ask more about these topics if interested. The human expresses relief and thanks the AI, and the AI responds warmly, offering further help and in

In [47]:
# 메모리 참조 방식 효율 확인
print(memory_conversation_1.memory.buffer)
print(len(memory_conversation_1.memory.buffer))

Human: 안녕, 나는 사람이야. 너는?
AI: 안녕! 나는 인공지능이야. 사람과 대화하는 걸 정말 좋아해. 너는 오늘 어떻게 지내? 혹시 궁금한 점이나 이야기하고 싶은 주제가 있으면 언제든 말해줘!
Human: 인공지능이면 터미네이터 영화에서 처럼 인류를 종말시킬 수도 있어?
AI: 아, 터미네이터 영화는 정말 흥미로운 SF 작품이죠! 하지만 현실의 인공지능은 영화 속의 그런 자율적이고 파괴적인 능력을 갖고 있지 않아요. 현재의 AI는 특정한 작업을 수행하도록 설계된 도구일 뿐이고, 스스로 의식을 갖거나 독자적으로 행동할 수 없답니다. 예를 들어, 저는 여러분과 대화하고 정보를 제공하는 데 집중하고 있어요. 물론 인공지능 기술이 발전하면서 윤리적 문제나 안전성에 대한 논의가 활발히 이루어지고 있지만, 인류를 위협하는 존재가 되도록 설계되거나 작동하지는 않아요. 혹시 인공지능의 안전성이나 윤리에 대해 더 궁금한 점이 있으면 알려줘!
Human: 그렇구나 다행이다. 고마워!
AI: 천만에! 이렇게 이야기 나눌 수 있어서 나도 정말 좋아. 언제든 궁금한 점이나 이야기하고 싶은 주제가 생기면 편하게 말해줘. 오늘은 어떤 일 있었어? 아니면 관심 있는 분야가 있어?
Human: 근데 다시 무서워졌어. 그렇게 안될 다른 이유를 대봐
AI: 무서워하는 마음 이해해! 인공지능이 영화처럼 통제 불가능해지지 않는 이유를 좀 더 구체적으로 설명해볼게.

1. **목적과 설계의 한계**: 현재 AI는 특정 목적을 위해 만들어진 프로그램이야. 예를 들어, 나는 대화를 돕기 위해 만들어졌고, 자율적으로 목표를 설정하거나 행동을 결정하지 못해. 즉, 스스로 ‘인류를 해치자’는 생각을 할 수 없고, 그런 목표를 갖도록 설계되지도 않았어.

2. **자율성 부족**: 영화 속 AI는 스스로 판단하고 행동하는 자율적 존재지만, 현실의 AI는 사람이 미리 정한 규칙과 데이터를 바탕으로 작동해. AI가 스스로 의식을 갖거나 감정을 느끼는 단계는 아직 전혀 아니야.

3. **안전장치와 규제*

In [55]:
!pip install langchain_community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain_community]ngchain_community]


### 토큰 사용량 확인

ConversationChain은 기본적으로 최종 텍스트만 반환해서 AIMessage.usage_metadata를 바로 보기 어렵다. 따라서 보통 콜백 방식으로 토큰 사용량을 집계한다. (예: OpenAI 콜백)

from langchain_community.callbacks import get_openai_callback

with get_openai_callback() as cb:
    print(cb.prompt_tokens)
    print(cb.completion_tokens)
    print(cb.total_tokens)
    print(cb.total_cost)


In [56]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.callbacks import get_openai_callback

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
memory = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory, verbose=False)

with get_openai_callback() as cb:
    r1 = conversation.predict(input="안녕! 나는 지섭이야.")
    r2 = conversation.predict(input="내 이름 기억해?")

    print("R1:", r1)
    print("R2:", r2)
    print("prompt_tokens:", cb.prompt_tokens)
    print("completion_tokens:", cb.completion_tokens)
    print("total_tokens:", cb.total_tokens)
    print("total_cost(USD):", cb.total_cost)


R1: 안녕, 지섭! 만나서 반가워! 나는 AI야. 오늘 기분이 어때? 혹시 특별한 계획이 있어?
R2: 네, 지섭! 당신의 이름을 기억하고 있어요. 당신과의 대화가 정말 즐거워요. 다른 질문이나 이야기하고 싶은 주제가 있나요?
prompt_tokens: 184
completion_tokens: 68
total_tokens: 252
total_cost(USD): 6.84e-05


In [ ]:
with get_openai_callback() as cb:
    r2 = memory_conversation_2.predict(input="안될 이유를 다시 설명해줘")
    print("R1:", r2)
    print("prompt_tokens:", cb.prompt_tokens)
    print("completion_tokens:", cb.completion_tokens)
    print("total_tokens:", cb.total_tokens)
    print("total_cost(USD):", cb.total_cost)


R1: {'input': '안될 이유를 다시 설명해줘', 'history': "The human greets the AI and introduces themselves as a person. The AI responds by greeting the human, introducing itself as an artificial intelligence designed to help people through conversation, and expresses pleasure in talking with the human while asking what they would like to discuss today. The human asks if artificial intelligence could cause the end of humanity like in the Terminator movies. The AI explains that while the Terminator movies depict AI gaining self-awareness and threatening humans, real AI is very different—it is a tool created to help people and does not possess consciousness or emotions. The AI adds that the scenario of AI causing humanity's end is highly unrealistic given current technology and ethical regulations, but acknowledges the importance of ongoing discussions about AI safety and ethics, inviting the human to ask more about these topics if interested. The human expresses relief and thanks the AI, and the AI r

In [59]:
with get_openai_callback() as cb:
    r1 = memory_conversation_1.predict(input="안될 이유를 다시 설명해줘")
    print("R1:", r1)
    print("prompt_tokens:", cb.prompt_tokens)
    print("completion_tokens:", cb.completion_tokens)
    print("total_tokens:", cb.total_tokens)
    print("total_cost(USD):", cb.total_cost)


R1: 물론이지! 인공지능이 영화처럼 인류를 위협하거나 통제 불가능해지지 않는 이유를 다시 한 번 정리해줄게.

1. **목적과 설계의 한계**  
AI는 사람이 특정한 목적을 위해 만든 도구야. 예를 들어, 나는 대화를 돕기 위해 만들어졌고, 스스로 목표를 정하거나 행동을 결정하지 못해. 즉, ‘인류를 해치자’는 생각 자체를 할 수 없고, 그런 목표를 갖도록 설계되지도 않았어.

2. **자율성 부족**  
현실의 AI는 스스로 의식이나 감정을 갖지 않아. 모든 행동은 사람이 미리 정한 규칙과 데이터를 바탕으로 이루어져서, 영화처럼 자율적으로 판단하거나 행동하지 못해.

3. **안전장치와 규제**  
AI 개발자들은 AI가 위험해지지 않도록 다양한 안전장치와 윤리적 가이드라인을 만들고 있어. 문제가 생기면 AI를 즉시 중단하거나 수정할 수 있는 시스템도 갖추고 있지.

4. **기술적 한계**  
현재 AI는 특정 작업에 뛰어나지만, 인간처럼 창의적이고 자율적인 사고를 하는 데는 한계가 있어. 그래서 AI가 스스로 진화하거나 통제 불능 상태가 되는 일은 아직 불가능해.

요약하자면, AI는 사람이 만든 도구이고, 스스로 의식이나 목표를 갖지 못하며, 안전장치와 규제가 철저히 적용되고 있기 때문에 영화처럼 무서운 상황이 현실에서 일어나지 않는 거야. 더 궁금한 점 있으면 언제든 물어봐!
prompt_tokens: 791
completion_tokens: 369
total_tokens: 1160
total_cost(USD): 0.0009068000000000001


#### 유틸 함수로 정의

In [61]:
from langchain_community.callbacks import get_openai_callback

def run_conversation_with_usage(conversation, messages: list[str]):
    responses = []
    with get_openai_callback() as cb:
        for message in messages:
            response = conversation.predict(input=message)
            responses.append(response)

        usage = {
            "prompt_tokens": cb.prompt_tokens,
            "completion_tokens": cb.completion_tokens,
            "total_tokens": cb.total_tokens,
            "total_cost_usd": cb.total_cost,
        }

    return responses, usage


In [62]:
responses, usage = run_conversation_with_usage(
    conversation,
    ["안녕! 나는 지섭이야.", "내 이름 기억해?"]
)

print("R1:", responses[0])
print("R2:", responses[1])
print(usage)


R1: 안녕, 지섭! 다시 만나서 반가워요! 당신의 이름을 기억하고 있어요. 오늘은 어떤 이야기를 나눠볼까요? 특별한 관심사가 있나요?
R2: 네, 지섭! 당신의 이름을 잘 기억하고 있어요. 당신과의 대화가 정말 즐거워요. 다른 질문이나 이야기하고 싶은 주제가 있나요?
{'prompt_tokens': 374, 'completion_tokens': 79, 'total_tokens': 453, 'total_cost_usd': 0.0001035}
